# Phase plot {#sec-phaseplot}

A phase plot represents the derivative $dX$ of a strategy $X$ against this same strategy $X$. The intersection of the resulting curve with the $x$-axis is a fixed point of the dynamics.

This tool is particularly useful to visualize convergence and equilibria along one dimension. In particular, it is suited to environments with only one agent and/or one state, such as a two-armed bandit task, that we use below as an example. 

First, we import what we need:

In [ ]:
 #| default_exp Utils.PhasePlot

In [ ]:
 #| hide
# Imports for the nbdev development environment
import nbdev
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
 #| export
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import itertools as it

from collections.abc import Callable

from pyDOE import lhs

from fastcore.utils import *

In [ ]:
# imports for testing
from pyDDRL.Environments.TwoArmedBandit import TwoArmedBandit
from pyCRLD.Agents.Base import abase
from pyCRLD.Agents.StrategySARSA import stratSARSA

## Implementation

In [ ]:
 #| export
def phaseplot(mae,  # CRLD multi-agent environment object
              x:tuple,  # which phase space axes to plot along x axes
              res,  # specify range & resolution
              NrRandom:int=3,  # how many random (in the other dimensions) stratgies for averaging 
              traj_len:int=500,  # trajectory length
              # col:str='LEN',  # color indicates either strength of flow via colormap, otherwise a fixed color name
              cmap='viridis',  # Colormap
              # kind='quiver+samples', # Kind of plot: "streamplot", "quiver+samples", "quiver", ...
              # sf=0.5,  # Scale factor for quiver arrows
              # lw=1.0,  # Line width for streamplot
              # dens=0.75,  # Density for streamplot
              acts=None,  # Action descriptions
              conds=None,  # Conditions descriptions 
              axes:Iterable=None,  # Axes to plot into
              verbose=False,  # shall I talk to you while working?
              ):  
    """
    Create a phase plot in strategy space.
    """
    
    # Fig and Axes
    if axes == None:
        states = mae.Z
        if mae.Z == 1:
            fig, ax = plt.subplots(1, states, figsize=(states * 3.8, 2.8))
            axes = [ax]
        else:
            fig, axes = plt.subplots(1, states, figsize=(states * 3.8, 2.8))
    else:
        axes = axes
        
    # cmap = mpl.cm.get_cmap(cmap)
    ext = np.linspace(0, 1, len(res))
    
    # x axis
    xs = res
    zero = np.zeros((len(xs)))

    # The Plots
    for i in range(len(x[1])): # go through each plot sequentially 
        xinds = (x[0][0], x[1][i], x[2][0])  # fix this later
        # obtain the data to plot
        RPEs, dXs = _data_to_plot(mae, res, xinds, traj_len, NrRandom, 
                             phasespace_items=_strategies, verbose=verbose)
        axes[i].plot(xs, RPEs[i, :], color="blue")
        axes[i].plot(xs, zero, color="grey", ls='dashed')
        twinax = axes[i].twinx()
        twinax.plot(xs, dXs[i, :], color="red")
        
        # Set default limits before aligning 
        axes[i].set_ylim(min(RPEs[i, :]), max(RPEs[i, :]))  # Primary y-axis
        twinax.set_ylim(min(dXs[i, :]), max(dXs[i, :]))     # Secondary y-axis
        if i==len(x[1])-1:
            twinax.set_ylabel("dX", color="red")

        # Align the zero levels of both axes
        align_zero_limits(axes[i], twinax)
        
    # Decorations
    # lens = max(xlens)
    if acts is None:
        acts = [f'act.{i}' for i in range(mae.M)]
        acts = mae.env.Aset
    if conds is None:
        conds = [f'state {i}' for i in range(mae.Z)]
    for i in range(len(x[1])):
        axes[i].set_title(conds[i])
        axes[i].set_xlabel("prob. act. "+str(xinds[2]))
    axes[0].set_ylabel("RPE", color="blue")
    
    # fig.tight_layout()
    
    return axes

In [ ]:
 #| export
# Function to adjust y-axis limits to ensure 0 is aligned
def align_zero_limits(ax1, ax2, padding_factor=0.05):
    # Get the limits of both y-axes
    ax1_min, ax1_max = ax1.get_ylim()
    ax2_min, ax2_max = ax2.get_ylim()

    # Compute ranges and negative-to-positive ratios
    ax1_range = max(ax1_max, 0) - min(ax1_min, 0)
    ax2_range = max(ax2_max, 0) - min(ax2_min, 0)
    
    ax1_neg_to_pos_ratio = -ax1_min / ax1_max if ax1_max != 0 else 1
    ax2_neg_to_pos_ratio = -ax2_min / ax2_max if ax2_max != 0 else 1
    
    # Adjust limits proportionally to keep 0 aligned
    if ax1_neg_to_pos_ratio > ax2_neg_to_pos_ratio:
        # Adjust ax2 to match ax1's ratio
        new_ax2_min = -ax2_max * ax1_neg_to_pos_ratio
        ax2.set_ylim(new_ax2_min, ax2_max)
        ax1.set_ylim(ax1_min )
    else:
        # Adjust ax1 to match ax2's ratio
        new_ax1_min = -ax1_max * ax2_neg_to_pos_ratio
        ax1.set_ylim(new_ax1_min, ax1_max)

    # Add padding (white space) to both axes
    ax1_min, ax1_max = ax1.get_ylim()
    ax2_min, ax2_max = ax2.get_ylim()

    # Extend the limits by a percentage (padding_factor)
    ax1_padding = (ax1_max - ax1_min) * padding_factor
    ax2_padding = (ax2_max - ax2_min) * padding_factor

    # Set the new padded limits
    ax1.set_ylim(ax1_min - ax1_padding, ax1_max + ax1_padding)
    ax2.set_ylim(ax2_min - ax2_padding, ax2_max + ax2_padding)

In [ ]:
 #| export
def _data_to_plot(mae,  # CRLD multi-agent environment object
                  res:Iterable,  # range & resolution of flow arrows 
                  xinds:tuple,  # of indices of the phase space object to plot along the x axis
                  traj_len:int,  # trajectory len
                  NrRandom:int,  # how many random (in the other dimensions) stratgies for averaging 
                  # difffunc:Callable,  # to compute which kind of arrows to plot (RPE or dX)
                  phasespace_items:Callable,  # to obtain phase space items for one ax plot point
                  verbose=False  # shall I talk to you while working?
                  ):
        
    l = len(res)
    RPEs = np.zeros((mae.Z, l))
    dXs = np.zeros((mae.Z, l))

    for i in range(mae.Z):
        for j, xval in enumerate(res):
            Xs = _strategies(mae, xinds, xval, NrRandom)  # generate random strategies to average over
            RPE_ = np.zeros((NrRandom))
            for k, xk in enumerate(Xs):
                prederror = mae.TDerror(xk, norm=True) # compute pred error
                RPE_[k] = prederror[xinds[0], i, xinds[2]]  # pred error for indices of interest
            RPEs[i, j] = np.mean(RPE_)  # add trajectory
            dX_ = np.array(_dXisa_s(Xs, mae))  # compute change in Xisa
            dX = np.mean(dX_, axis=0)  # average over random strategies
            dXs[i, j] = dX[xinds[0], i, xinds[2]]  # dX for indices of interest
            
    return RPEs, dXs

In [ ]:
 #| export
def _dXisa_s(Xisa_s:Iterable, # of joint strategies `Xisa`
            mae  # CRLD multi-agent environment object
            ) -> np.ndarray:  # joint strategy differences 
    """Compute `Xisa`(t+1)-`Xisa`(t) for all `Xisa_s`."""
    return np.array([mae.step(Xisa)[0] - Xisa for Xisa in Xisa_s])

In [ ]:
 #| export
def _strategies(mae,  # CRLD multi-agent environment object
                xinds:tuple,  # of indices of the phase space item to plot along the x axis
                xval:float,  # the value of the phase space item to plot along the x axis
                NrRandom,  # how many random (in the other dimensions) stratgies for averaging
               ) -> np.ndarray: # Array of joint strategies
    """
    Creates strategies (as a particular type of phase space item) for one ax plot point.
    All strategies have value `xval` at the `xinds` index and value `yval` at the `yinds`. 
    """   
    N, C, M = mae.N, mae.Q, mae.M  # Number of agents, conditions, actions
    # Xs = np.random.rand(NrRandom, N, C, M)  # random policies
    Xs = lhs(N*C*M, NrRandom).reshape(NrRandom, N, C, M)  # using latin hypercube sampling
    Xs = Xs / Xs.sum(axis=-1, keepdims=True)  # properly normalised
    
    xi, xc, xa = xinds; xa_ = tuple(set(range(M)) - set([xa]))
    
    Xs[:, xi, xc, xa] = xval  # setting x and y values
    
    # normalisation
    Xs[:,xi,xc,xa_] = (1-Xs[0, xi, xc, xa]) * Xs[:,xi,xc,xa_]\
        / np.sum(Xs[:,xi,xc,xa_], axis=-1, keepdims=True)
    
    return Xs

## Example with a two-armed bandit task

Let's consider a two-armed bandit task with $p_1 = 0.7$ and $p_2 = 0.3$. 

In [ ]:
# Define environment
xs = np.array([[-1, 1],
               [-1, 1]])
ps = np.array([[0.3, 0.7],
               [0.7, 0.3]])
tab = TwoArmedBandit(disttype='discrete', xs=xs, ps=ps)

Now, we define an agent-environment interface comprised of a SARSA agent playing the aforementioned task. This agent has a learning rate $\alpha=0.1$, a discount factor $\gamma = 0$, and a choice intensity $\beta = 4$. 

In [ ]:
# Agent-environment interface
aei = stratSARSA(tab,
                 learning_rates=0.1,
                 discount_factors=0,
                 choice_intensities=4.,
                 use_prefactor=False)

In [ ]:
x = ([0], [0], [0])  # (i,s,a) to plot along the x-axis
res_phase = np.linspace(0, 1., 51)  # resolution for phase plot
res_phase[0] = 0.001
res_phase[-1] = 0.999

Now making the phase plot:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 3))
phaseplot(aei, x, res_phase, NrRandom=1, axes=[ax])
ax.set_xlabel("strategy "+r'$X$')
ax.set_title("phase plot")
plt.show()

<Figure size 1200x900 with 2 Axes>

In [ ]:
 #| hide
import nbdev; nbdev.nbdev_export()